# Cross-Template Key-Value Field Extraction — Kaggle Run 2

**IE 643 Course Project** · Joint Key-Value Extraction, VRDU Ad-Buy & Registration Generalization, and Synthetic AcroForm Pre-training

| Experiment | Dataset / Subcorpus | Protocol / Task | Objective |
|---|---|---|---|
| **Track A+** | FUNSD | Joint (Tagging + Linking) | Evaluate multi-task entity F1 and spatial biaffine linking F1 |
| **Track B1** | VRDU `registration-form` | Matched MTL vs UTL | Generalization gap across 21 government FARA form templates |
| **Track B2** | VRDU `ad-buy-form` | Matched MTL vs UTL | Generalization gap across 10 commercial FCC political TV ad templates |
| **Track C** | Synthetic AcroForm + Augraphy | Multi-Template Joint | Programmatic multi-template training engine with realistic scan noise |

> **Hardware**: Kaggle GPU (Tesla P100 16 GB or 2×T4).  
> **Weekly Quota**: 30 hours.

## 0 · Environment Setup & Dependencies

In [ ]:
# ── Clone or Pull KeyValue-Extraction Repository ─────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/ABasakEE/KeyValue-Extraction.git"
REPO_DIR = "/kaggle/working/KeyValue-Extraction"

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"{REPO_DIR} already exists, pulling latest main branch...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Add src/ to sys.path
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print(f"sys.path: {sys.path[:2]}")

In [ ]:
# ── Install Required Dependencies ───────────────────────────────────────
!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval faker augraphy

In [ ]:
# ── Verify GPU and Environment ──────────────────────────────────────────
import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Fine-tuning on CPU is prohibitively slow.")

## 1 · Download Datasets

In [ ]:
# ── Download Raw FUNSD (with full key-value linking annotations) ────────
FUNSD_DIR = os.path.join(REPO_DIR, "data", "funsd", "dataset")

if not os.path.isdir(FUNSD_DIR):
    print("Downloading official raw FUNSD release...")
    !mkdir -p data/funsd
    !wget -q https://guillaumejaume.github.io/FUNSD/dataset.zip -O data/funsd/dataset.zip
    !cd data/funsd && unzip -q dataset.zip && rm dataset.zip
    print("FUNSD downloaded and extracted successfully.")
else:
    print(f"FUNSD already present at {FUNSD_DIR}")

# Verify paths
for split in ("training_data", "testing_data"):
    for sub in ("annotations", "images"):
        p = os.path.join(FUNSD_DIR, split, sub)
        n = len(os.listdir(p)) if os.path.isdir(p) else 0
        print(f"  {split}/{sub}: {n} files")

In [ ]:
# ── Clone Google Research VRDU (Registration Forms + Ad-Buy Forms) ─────
VRDU_DIR = os.path.join(REPO_DIR, "data", "vrdu")

if not os.path.isdir(VRDU_DIR):
    print("Cloning VRDU benchmark repository...")
    !git clone --depth 1 https://github.com/google-research-datasets/vrdu.git data/vrdu
    print("VRDU cloned successfully.")
else:
    print(f"VRDU already present at {VRDU_DIR}")

# Verify both registration-form and ad-buy-form subcorpora
for subcorp in ("registration-form", "ad-buy-form"):
    base = os.path.join(VRDU_DIR, subcorp)
    data_gz = os.path.join(base, "main", "dataset.jsonl.gz")
    meta = os.path.join(base, "main", "meta.json")
    splits = os.path.join(base, "few_shot-splits")
    n_splits = len(os.listdir(splits)) if os.path.isdir(splits) else 0
    print(f"  {subcorp}: data={'OK' if os.path.isfile(data_gz) else 'MISSING'} | "
          f"meta={'OK' if os.path.isfile(meta) else 'MISSING'} | "
          f"splits={n_splits} files")

## 2 · Synthetic Form Generation (AcroForm + Augraphy)

In [ ]:
# ── Generate Multi-Template Synthetic Form Dataset ──────────────────────
# Generates 200 training and 50 testing documents across diverse template
# families (Invoices, Medical Intake, FCC Ad-Buy, FARA Disclosure) with
# Augraphy scanner noise, paper texture, and physical photocopy artifacts.

!python src/data/synthetic_generator.py --n_train 200 --n_test 50 --output_dir data/synthetic/dataset

---
## 3 · Track A+ — Joint Entity Tagging & Key-Value Linking on FUNSD

**Objective**: End-to-end multi-task learning optimizing token classification ($\mathcal{L}_{\text{tag}}$) and spatial biaffine relation extraction ($\mathcal{L}_{\text{link}}$).
Evaluates both **Entity F1** and **Linking F1** across 3 random seeds.

In [ ]:
# ── Run Track A+ (Joint LiLT on FUNSD) ──────────────────────────────────
!python src/train.py --config configs/funsd_joint_lilt.yaml

---
## 4 · Track B1 — VRDU Registration Forms: Matched-Size MTL vs UTL

**Objective**: Measure the generalization gap on government foreign agent disclosure forms across 21 templates at matched training-set size ($N_{\text{train}}=200$).

In [ ]:
# ── Run Track B1 (VRDU Registration Forms) ──────────────────────────────
!python src/train.py --config configs/vrdu_lilt.yaml --protocol matched

---
## 5 · Track B2 — VRDU Ad-Buy Forms: Matched-Size MTL vs UTL

**Objective**: Measure the cross-template generalization gap on commercial broadcast political TV advertising contracts (FCC DeepForm) across 10 distinct templates at matched training size ($N_{\text{train}}=200$).

In [ ]:
# ── Run Track B2 (VRDU Ad-Buy Forms) ────────────────────────────────────
!python src/train.py --config configs/vrdu_adbuy_lilt.yaml --protocol matched

---
## 6 · Track C — Joint Extraction on Synthetic Multi-Template Forms

**Objective**: Train and evaluate joint token extraction and spatial biaffine relation linking on the synthetic multi-template AcroForm + Augraphy dataset.

In [ ]:
# ── Run Track C (Synthetic Multi-Template Dataset) ───────────────────────
!python src/train.py --config configs/synthetic_joint_lilt.yaml

---
## 7 · Summary of Results & Generalization Gaps

In [ ]:
import json
from pathlib import Path

def print_summary(path, label):
    p = Path(path)
    if not p.exists():
        print(f"[-] {label}: No results found at {path}")
        return
    data = json.loads(p.read_text(encoding='utf-8'))
    print(f"\n{'=' * 65}")
    print(f"[+] {label}")
    print(f"{'=' * 65}")
    if 'gap' in data:
        gap = data['gap']
        print(f"  MTL Mean F1: {gap.get('mtl_mean_f1', 0):.4f} +/- {gap.get('mtl_std', 0):.4f}")
        print(f"  UTL Mean F1: {gap.get('utl_mean_f1', 0):.4f} +/- {gap.get('utl_std', 0):.4f}")
        print(f"  Generalization Gap: {gap.get('gap_f1', 0):.4f} ({gap.get('relative_drop_pct', 0):.1f}% relative drop)")
    else:
        print(f"  Mean Entity F1:  {data.get('mean_f1', 0):.4f} +/- {data.get('std_f1', 0):.4f}")
        if 'mean_linking_f1' in data:
            print(f"  Mean Linking F1: {data.get('mean_linking_f1', 0):.4f} +/- {data.get('std_linking_f1', 0):.4f}")

print_summary("results/funsd_joint_lilt/metrics.json", "Track A+ : Joint FUNSD (Tagging + Linking)")
print_summary("results/vrdu_lilt/metrics.json", "Track B1 : VRDU Registration Forms (MTL vs UTL)")
print_summary("results/vrdu_adbuy_lilt/metrics.json", "Track B2 : VRDU Ad-Buy Forms (MTL vs UTL)")
print_summary("results/synthetic_joint_lilt/metrics.json", "Track C  : Synthetic AcroForms (Joint)")